In [23]:
import pandas as pd

file_path = "/content/online_retail_II.xlsx"

df = pd.read_excel(file_path)

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
display(df.head())

Dataset Shape: (525461, 8)

Columns:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

First 5 Rows:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [24]:
df_clean = df.copy()

df_clean = df_clean.dropna(subset=['Customer ID'])

df_clean = df_clean[~df_clean['Invoice'].astype(str).str.startswith('C')]

df_clean = df_clean[
    (df_clean['Quantity'] > 0) &
    (df_clean['Price'] > 0)
]

df_clean['Order_Value'] = df_clean['Quantity'] * df_clean['Price']

print("Original Shape:", df.shape)
print("Cleaned Shape:", df_clean.shape)

print("\nMissing Customer IDs:", df_clean['Customer ID'].isna().sum())
print("Cancelled Invoices:", df_clean['Invoice'].astype(str).str.startswith('C').sum())
print("Invalid Quantity:", (df_clean['Quantity'] <= 0).sum())
print("Invalid Price:", (df_clean['Price'] <= 0).sum())

Original Shape: (525461, 8)
Cleaned Shape: (407664, 9)

Missing Customer IDs: 0
Cancelled Invoices: 0
Invalid Quantity: 0
Invalid Price: 0


In [25]:
customer_summary = df_clean.groupby('Customer ID').agg(
    Total_Orders=('Invoice', 'nunique'),
    Total_Sales=('Order_Value', 'sum')
).reset_index()

customer_summary['Segment'] = customer_summary['Total_Orders'].apply(
    lambda x: 'Repeat Customer' if x >= 2 else 'One-Time Customer'
)

total_customers = customer_summary['Customer ID'].nunique()
repeat_customers = (customer_summary['Segment'] == 'Repeat Customer').sum()
one_time_customers = (customer_summary['Segment'] == 'One-Time Customer').sum()

repeat_rate = (repeat_customers / total_customers) * 100

print("Total Customers:", total_customers)
print("Repeat Customers:", repeat_customers)
print("One-Time Customers:", one_time_customers)
print("Repeat Purchase Rate: {:.2f}%".format(repeat_rate))

segment_summary = customer_summary.groupby('Segment').agg(
    Customers=('Customer ID', 'count'),
    Total_Orders=('Total_Orders', 'sum'),
    Total_Sales=('Total_Sales', 'sum')
).reset_index()

segment_summary['AOV'] = (
    segment_summary['Total_Sales'] / segment_summary['Total_Orders']
)

segment_summary['Customer_Percentage'] = (
    segment_summary['Customers'] / total_customers * 100
)

segment_summary['Total_Sales'] = segment_summary['Total_Sales'].round(2)
segment_summary['AOV'] = segment_summary['AOV'].round(2)
segment_summary['Customer_Percentage'] = segment_summary['Customer_Percentage'].round(2)

print("\nCustomer Segment Table:")
display(segment_summary)

Total Customers: 4312
Repeat Customers: 2893
One-Time Customers: 1419
Repeat Purchase Rate: 67.09%

Customer Segment Table:


,Segment,Customers,Total_Orders,Total_Sales,AOV,Customer_Percentage
0,One-Time Customer,1419,1419,499041.79,351.69,32.91
1,Repeat Customer,2893,17794,8332961.48,468.30,67.09


In [26]:
top_repeat_customers = customer_summary[
    customer_summary['Segment'] == 'Repeat Customer'
].sort_values('Total_Orders', ascending=False).head(10)

print("Top 10 Repeat Customers:")
display(top_repeat_customers)

Top 10 Repeat Customers:


,Customer ID,Total_Orders,Total_Sales,Segment
1840,14911.0,205,152147.57,Repeat Customer
3996,17850.0,155,51208.87,Repeat Customer
251,12748.0,144,22879.66,Repeat Customer
2133,15311.0,121,56003.26,Repeat Customer
506,13089.0,109,57912.03,Repeat Customer
1269,14156.0,102,196566.74,Repeat Customer
1606,14606.0,102,18704.41,Repeat Customer
939,13694.0,94,131443.19,Repeat Customer
3989,17841.0,91,29892.50,Repeat Customer
4183,18102.0,89,349164.35,Repeat Customer


In [27]:
df_clean.to_csv('/content/Cleaned_Online_Retail_II.csv', index=False)

print("Cleaned dataset exported successfully.")

Cleaned dataset exported successfully.


Key Insights

1. High repeat purchase rate: 67.09% of the 4,312 customers are repeat customers, showing that a large portion of customers made purchases on two or more distinct invoices.
2. Repeat customers generate most sales: Repeat customers generated 8,332,961.48 in sales across 17,794 orders, compared with 499,041.79 from one-time customers.
3. Repeat customers have higher AOV: Repeat customers have an average order value of 468.30, compared with 351.69 for one-time customers. This shows that repeat customers generally place higher-value orders.
4. Repeat customers dominate order activity: Although repeat customers represent 67.09% of the customer base, they account for 17,794 orders, indicating substantially greater purchasing activity than one-time customers.
5. A small group of customers purchases very frequently: The top repeat customer, Customer ID 14911, made 205 orders, while several other customers made more than 100 orders. This indicates the presence of highly active repeat buyers.


Main takeaway

The analysis shows that repeat customers are an important contributor to both order volume and sales. Tracking repeat purchase rate and AOV can therefore help businesses understand customer loyalty and identify opportunities to encourage one-time customers to make additional purchases.